In [0]:
from pyspark.sql.functions import col, explode_outer, current_timestamp

# 1. Setup Widgets
dbutils.widgets.text("storage_name", "miniproject123")
dbutils.widgets.text("key_scope", "secret-scope")
storage_name = dbutils.widgets.get("storage_name")
key_scope = dbutils.widgets.get("key_scope")

# 2. Configure ADLS Gen2 Authentication
tenant_id     = dbutils.secrets.get(scope=key_scope, key="client-tenant")
client_id     = dbutils.secrets.get(scope=key_scope, key="client-secret")
client_secret = dbutils.secrets.get(scope=key_scope, key="client-value")
spark.conf.set(f"fs.azure.account.auth.type.{storage_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# 3. Paths Configuration
raw_input_path = f"abfss://bronze@{storage_name}.dfs.core.windows.net/ingested_products/"
silver_output_path = f"abfss://silver@{storage_name}.dfs.core.windows.net/dim_products/"

# 3. Read Raw Delta Lake Table
df_raw = spark.read.format("delta").load(raw_input_path)

# 4. Cleansing & Unpacking Nested Structs
df_silver = df_raw.select(
    col("id").cast("integer").alias("product_id"),
    col("title").alias("product_name"),
    col("description"),
    col("category"),
    col("brand"),
    col("price").cast("double"),
    col("discountPercentage").cast("double").alias("discount_percentage"),
    col("rating").cast("double"),
    col("stock").cast("integer"),
    # Unpacking nested struct 'dimensions'
    col("dimensions.width").cast("double").alias("dim_width_cm"),
    col("dimensions.height").cast("double").alias("dim_height_cm"),
    col("dimensions.depth").cast("double").alias("dim_depth_cm"),
    # Unpacking nested struct 'meta'
    col("meta.barcode").alias("barcode"),
    col("meta.qrCode").alias("qr_code_url"),
    col("availabilityStatus").alias("availability_status"),
    col("returnPolicy").alias("return_policy"),
    col("ingestion_timestamp").alias("raw_ingestion_timestamp"),
    current_timestamp().alias("silver_processed_timestamp")
).dropDuplicates(["product_id"])

# 5. Write Cleansed Data into Silver Container in Delta Format
df_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .save(silver_output_path)

# 6. Verify Silver Output
display(spark.read.format("delta").load(silver_output_path))